<a href="https://colab.research.google.com/github/rdiazrincon/conformal_prediction_pd/blob/master/pd_kth_record.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PD LEDD Change Prediction — K-th Record Version

Predicts LEDD change at the **K-th hospital visit** across patients, rather than
across all time windows. This restructuring aligns the prediction task with the
exchangeability assumption required by conformal prediction: for a fixed K,
observations are one-per-patient and therefore independent.

**Set `K` in the constants cell.** For K=1 all changes are 0 (no predecessor) —
use K≥2 for a meaningful prediction task.

**Files required in `data/`:**
`CROSSOVER_2.csv` · `DEMOGRAPHICS.csv` · `DIAGNOSIS_DATE_3.csv` · `DBS_pts.csv`


## 1 · Imports and constants

In [1]:
import numpy as np
import pandas as pd
import json, re, warnings
from datetime import datetime
from scipy import stats
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    roc_curve, mean_squared_error, mean_absolute_error, r2_score,
)
warnings.filterwarnings('ignore')

RANDOM_STATE = 21
DATA_CUTOFF  = datetime(2021, 12, 31)
K            = 1   # ← change this: 1=first visit, 2=second, 3=third …
               # K=1 → all changes=0 (no predecessor); use K≥2 for real predictions
print(f"Running K={K} version")


Running K=1 version


## 2 · Load raw files

In [2]:
# CROSSOVER_2.csv is used as the primary drug source — it contains the full
# visit history (outpatient + inpatient) with visit_start_datetime, giving
# meaningful temporal spread per patient.
drug_exposure = pd.read_csv(
    'data/CROSSOVER_2.csv',
    dtype={'dose_unit_source_value': str},
    engine='python',
    on_bad_lines='warn',
)
drug_exposure['drug_exposure_start_datetime'] = pd.to_datetime(
    drug_exposure['drug_exposure_start_datetime'])
drug_exposure['visit_start_datetime'] = pd.to_datetime(
    drug_exposure['visit_start_datetime'])
# dose_source_value not present in CROSSOVER_2 — calculate_led handles NaN gracefully
drug_exposure['dose_source_value']      = np.nan
drug_exposure['dose_unit_source_value'] = np.nan
drug_exposure['route_source_value']     = np.nan
drug_exposure['visit_detail_id']        = np.nan

demographics = pd.read_csv('data/DEMOGRAPHICS.csv')
demographics['birth_datetime'] = pd.to_datetime(demographics['birth_datetime'])
demographics['age'] = ((pd.Timestamp('now') - demographics['birth_datetime']).dt.days / 365.25).astype(int)
demographics.drop(columns=['birth_datetime'], inplace=True)
demographics = demographics[['person_id','age','gender_source_value',
                              'race_source_value','ethnicity_source_value']]

diagnosis_date = pd.read_csv('data/DIAGNOSIS_DATE_3.csv')
diagnosis_date['diagnosis_date'] = pd.to_datetime(diagnosis_date['diagnosis_date'])

visit_occurrence = pd.read_csv('data/CROSSOVER_2.csv', engine='python', on_bad_lines='warn')
visit_occurrence['visit_start_datetime'] = pd.to_datetime(visit_occurrence['visit_start_datetime'])

dbs_df = pd.read_csv('data/DBS_pts.csv')
dbs_df['procedure_date']       = pd.to_datetime(dbs_df['procedure_date'])
dbs_df['condition_start_date'] = pd.to_datetime(dbs_df['condition_start_date'])

print(f"drug_exposure    : {drug_exposure['person_id'].nunique()} patients, {len(drug_exposure):,} rows")
print(f"demographics     : {len(demographics):,} rows")
print(f"visit_occurrence : {visit_occurrence['person_id'].nunique()} patients")
print(f"dbs_df           : {dbs_df['person_id'].nunique()} DBS patients")


drug_exposure    : 631 patients, 554,590 rows
demographics     : 2,724 rows
visit_occurrence : 631 patients
dbs_df           : 142 DBS patients


## 3 · Parse drug names and dosages

In [3]:
def safe_parse_dsv(val):
    try:
        parsed = json.loads(val)
        name   = parsed.get('med_display_name', '')
        return name if isinstance(name, str) else ''
    except Exception:
        return ''

dsv_drug_exposure           = drug_exposure['drug_source_value'].apply(safe_parse_dsv)
drug_info_drug_source_value = {i: v for i, v in enumerate(dsv_drug_exposure)}
drugs_used_drug_exposure    = [v.lower() for v in drug_info_drug_source_value.values()]

drug_names_pattern = r"([\w\s-]+)\s(?:\(([\w\s-]+)\)\s*)?"
dosage_pattern     = r"\d+(?:\.\d+)?(?:-\d+(?:\.\d+)?)*(?:\s*(?:mg/ml|mg|ml))(?:/hr)?"

generic_names_de, brand_names_de, dosages_de = [], [], []
for string in drugs_used_drug_exposure:
    m = re.findall(drug_names_pattern, string)
    if m:
        g, b = m[0]
        generic_names_de.append(g)
        brand_names_de.append(b if b else np.nan)
        dm = re.findall(dosage_pattern, string)
        dosages_de.append(dm[0] if dm else np.nan)
    else:
        generic_names_de.append(np.nan)
        brand_names_de.append(np.nan)
        dosages_de.append(np.nan)

pd_data_drug_exposure = pd.DataFrame({
    'generic_name': generic_names_de,
    'brand_name':   brand_names_de,
    'dosage':       dosages_de,
})
for bad, good in [('1mg','1 mg'),('200mg','200 mg')]:
    idx = pd_data_drug_exposure[pd_data_drug_exposure['dosage']==bad].index
    pd_data_drug_exposure.loc[idx,'dosage'] = good

print(f"pd_data_drug_exposure: {len(pd_data_drug_exposure):,} rows, "
      f"{pd_data_drug_exposure['generic_name'].nunique()} unique drug names")


pd_data_drug_exposure: 554,590 rows, 1480 unique drug names


## 4 · Build `led_df` and calculate LED

In [4]:
led_dose = []
for item in pd_data_drug_exposure['dosage']:
    if isinstance(item, float) and np.isnan(item):
        led_dose.append('0')
    elif '-' in str(item):
        led_dose.append(str(item).split('-')[1].split()[0])
    else:
        led_dose.append(str(item).split()[0])

led_df = pd.concat(
    [drug_exposure.iloc[:,0:3], pd_data_drug_exposure, drug_exposure.iloc[:,3:]], axis=1
)
led_df.insert(loc=6, column='led_dose',  value=led_dose)
led_df.insert(loc=3, column='drug_info', value=drug_info_drug_source_value)
led_df['led_dose']                     = pd.to_numeric(led_df['led_dose'], errors='coerce')
led_df['drug_exposure_start_datetime'] = pd.to_datetime(led_df['drug_exposure_start_datetime'])
led_df['generic_name'].replace({np.nan:'None'}, inplace=True)
led_df['brand_name'].replace({np.nan:'None'},   inplace=True)

conversion_factors = {
    'amantadine':1.0,           'amantadine er':1.25,
    'apomorphine':10.0,         'benztropine':1.0,
    'benztropine mesylate':1.0, 'bromocriptine':10.0,
    'cabergoline':66.7,
    'carbidopa-levodopa':1.0,
    'inv carbidopa-levodopa intestinal gel':1.0,
    'inv carbidopa-levodopa intestinal gel pump':1.0,
    'carbidopa':0.1,            'carbidopa-levodopa er':0.5,
    'carbidopa-levodopa-entacapone':1.33, 'entacapone':1.33,
    'pramipexole':100.0,        'pramipexole er':100.0,
    'trihexyphenidyl':1.0,      'rasagiline':100.0,
    ' rasagiline mesylate':100.0,
    'ropinirole':0.5,           'rotigotine':30.0,
    'selegiline':10.0,          'tolcapone':1.5,
}

def calculate_led(row):
    dsv = row['dose_source_value']
    ld  = row['led_dose']
    cf  = conversion_factors.get(row['generic_name'], 0)
    if ld == dsv:
        dsv = 1.0
    if dsv == 0.0:
        return None
    return dsv * ld * cf if pd.notna(dsv) else ld * cf

led_df['led'] = led_df.apply(calculate_led, axis=1)

# Set LED=NaN for non-PD drugs so they become empty resample buckets
# that get forward-filled — preserves temporal coverage from all visits
pd_drug_names = set(conversion_factors.keys())
led_df.loc[~led_df['generic_name'].isin(pd_drug_names), 'led'] = np.nan

print(f"led_df          : {led_df['person_id'].nunique()} patients, {len(led_df):,} rows")
print(f"PD drug rows    : {led_df['led'].notna().sum():,}")
print(f"Non-PD (NaN)    : {led_df['led'].isna().sum():,}")


led_df          : 631 patients, 554,590 rows
PD drug rows    : 47,449
Non-PD (NaN)    : 507,141


## 5 · Sort `led_df`

CROSSOVER_2.csv already provides `visit_start_datetime` — no additional merge needed.

In [5]:
led_df.sort_values(by='visit_start_datetime', ascending=True, inplace=True)

print(f"led_df: {led_df['person_id'].nunique()} patients, {len(led_df):,} rows")
print(f"visit_start_datetime: {led_df['visit_start_datetime'].min().date()} → "
      f"{led_df['visit_start_datetime'].max().date()}")
unique_years = led_df.groupby('person_id')['visit_start_datetime'].apply(
    lambda x: x.dt.year.nunique())
print(f"Unique years per patient: min={unique_years.min()}  "
      f"median={unique_years.median():.0f}  max={unique_years.max()}")


led_df: 631 patients, 554,590 rows
visit_start_datetime: 2011-05-16 → 2021-04-29
Unique years per patient: min=1  median=4  max=11


## 6 · DBS setup

In [6]:
cpt_codes = [
    '95970','95961','95962','61867','61885','95972','95978','95983','95979','61868',
    '95984','95974','61886','L8681','95971','61888','61880','C1787','95973',
    '00H03MZ','0NH00NZ','00W03MZ','0JWT0MZ','0JPT0MZ','00P00MZ','00W00MZ',
    '00P03MZ','00H00MZ','0JH60BZ','0JH60DZ','00H04MZ',
]
icd_codes = ['Z96.82','T85.110A']

procedures = dbs_df['procedure_source_value'].copy().fillna('None')
conditions = dbs_df['condition_source_value'].copy().fillna('None')
for val in procedures.unique():
    for code in cpt_codes:
        if code in str(val): procedures.replace(val, code, inplace=True)
for val in conditions.unique():
    for code in icd_codes:
        if code in str(val): conditions.replace(val, code, inplace=True)
dbs_df['procedure_source_value'] = procedures
dbs_df['condition_source_value'] = conditions

dbs_pts    = dbs_df['person_id'].unique().tolist()
all_pts    = led_df['person_id'].unique().tolist()
has_dbs_df = pd.DataFrame({
    'person_id': all_pts,
    'has_dbs':   [1 if pt in dbs_pts else 0 for pt in all_pts],
})
print(f"DBS patients: {has_dbs_df['has_dbs'].sum()} / {len(has_dbs_df)}")


DBS patients: 142 / 631


## 7 · K-th record selection

Replaces the time-window resample. Each patient contributes exactly one row — their K-th hospital visit ordered chronologically.

In [7]:
# Average LED across all drug records within each unique visit per patient
visit_led = (
    led_df.groupby(['person_id','visit_start_datetime'])['led']
          .mean()
          .reset_index()
          .rename(columns={'led':'mean_led_per_visit'})
)
visit_led.sort_values(['person_id','visit_start_datetime'], inplace=True)
visit_led['visit_rank'] = visit_led.groupby('person_id').cumcount() + 1

print(f"Total unique visits: {len(visit_led):,}")
print(f"Visits per patient: min={visit_led.groupby('person_id').size().min()}  "
      f"median={visit_led.groupby('person_id').size().median():.0f}  "
      f"max={visit_led.groupby('person_id').size().max()}")

# Select K-th visit per patient — patients with fewer than K visits are excluded
kth = visit_led[visit_led['visit_rank'] == K].copy()
kth.drop(columns=['visit_rank'], inplace=True)

# Compute LEDD change from visit (K-1) to visit K
if K > 1:
    prev = (
        visit_led[visit_led['visit_rank'] == K - 1]
                 [['person_id','mean_led_per_visit']]
                 .rename(columns={'mean_led_per_visit':'prev_led'})
    )
    kth = kth.merge(prev, on='person_id', how='left')
    kth['change_in_ledd'] = kth['mean_led_per_visit'] - kth['prev_led']
    kth['percent_change']  = (kth['change_in_ledd'] / kth['prev_led']) * 100
    kth.drop(columns=['prev_led'], inplace=True)
else:
    kth['change_in_ledd'] = 0.0
    kth['percent_change']  = 0.0

mean_led_per_visit = kth.reset_index(drop=True)

print(f"\nK={K}: {len(mean_led_per_visit):,} patients have a K-th visit")
print(f"visit_start_datetime range: "
      f"{mean_led_per_visit['visit_start_datetime'].min().date()} → "
      f"{mean_led_per_visit['visit_start_datetime'].max().date()}")
print(f"mean_led_per_visit: min={mean_led_per_visit['mean_led_per_visit'].min():.1f}  "
      f"mean={mean_led_per_visit['mean_led_per_visit'].mean():.1f}  "
      f"max={mean_led_per_visit['mean_led_per_visit'].max():.1f}")


Total unique visits: 8,519
Visits per patient: min=1  median=8  max=837

K=1: 631 patients have a K-th visit
visit_start_datetime range: 2011-05-16 → 2021-03-01
mean_led_per_visit: min=0.1  mean=103.6  max=257.0


## 8 · Build target variables

In [8]:
mean_led_per_visit['is_first_visit'] = (mean_led_per_visit['change_in_ledd'] == 0.0)

def normalize_percent_change(series):
    zero_mask   = series == 0
    non_zero    = series[~zero_mask]
    if len(non_zero) == 0:
        return pd.Series(0.0, index=series.index)
    transformed = np.sign(non_zero) * np.log1p(np.abs(non_zero))
    normalized  = stats.mstats.winsorize(transformed, limits=[0.05, 0.05])
    normalized  = (normalized - normalized.min()) / (normalized.max() - normalized.min()) * 2 - 1
    result = pd.Series(index=series.index, dtype=float)
    result[zero_mask]  = 0.0
    result[~zero_mask] = normalized
    return result

mean_led_per_visit['normalized_percent_change'] = normalize_percent_change(
    mean_led_per_visit['percent_change']
).round(2)
mean_led_per_visit['is_change']  = (mean_led_per_visit['normalized_percent_change'] != 0).astype(int)
mean_led_per_visit['prediction'] = mean_led_per_visit['is_change']

normalized_pctg_change = mean_led_per_visit['normalized_percent_change'].copy()

print(f"Class balance → change: {mean_led_per_visit['prediction'].mean():.1%}  "
      f"| no-change: {1-mean_led_per_visit['prediction'].mean():.1%}")
print(f"Non-zero normalized_percent_change: "
      f"{(mean_led_per_visit['normalized_percent_change']!=0).sum()}")


Class balance → change: 0.0%  | no-change: 100.0%
Non-zero normalized_percent_change: 0


## 9 · Feature merges

In [9]:
# DBS flag
mean_led_per_visit = mean_led_per_visit.merge(has_dbs_df, on='person_id', how='left')
mean_led_per_visit.sort_values(['person_id','visit_start_datetime'],
                                ascending=[False,True], inplace=True)

# Demographics
mean_led_per_visit = mean_led_per_visit.merge(demographics, on='person_id', how='inner')

# Length of stay + days since last visit (computed from full visit history)
los = (
    visit_occurrence.groupby('person_id')['visit_start_datetime']
                    .agg(['min','max']).reset_index()
)
los['length_of_stay']        = ((los['max'] - los['min']) / np.timedelta64(1,'D')).astype(int)
los['days_since_last_visit'] = ((DATA_CUTOFF - los['max']) / np.timedelta64(1,'D')).astype(int)
los.drop(columns=['min','max'], inplace=True)
mean_led_per_visit = mean_led_per_visit.merge(los, on='person_id', how='left')
mean_led_per_visit['length_of_stay'].fillna(los['length_of_stay'].mean(),             inplace=True)
mean_led_per_visit['days_since_last_visit'].fillna(los['days_since_last_visit'].mean(), inplace=True)

# Days to PD diagnosis
first_visit = visit_occurrence.groupby('person_id')['visit_start_datetime'].agg('min').reset_index()
first_diag  = diagnosis_date.groupby('person_id')['diagnosis_date'].agg('min').reset_index()
days_to_dx  = first_visit.merge(first_diag, on='person_id', how='left')
days_to_dx['days_to_diagnosis'] = (
    (days_to_dx['diagnosis_date'] - days_to_dx['visit_start_datetime'])
    / np.timedelta64(1,'D')
)
days_to_dx['days_to_diagnosis'].fillna(days_to_dx['days_to_diagnosis'].mean(), inplace=True)
days_to_dx['days_to_diagnosis'] = days_to_dx['days_to_diagnosis'].astype(int)
days_to_dx = days_to_dx[['person_id','days_to_diagnosis']]
mean_led_per_visit = mean_led_per_visit.merge(days_to_dx, on='person_id', how='left')
mean_led_per_visit['days_to_diagnosis'].fillna(days_to_dx['days_to_diagnosis'].mean(), inplace=True)

# Days to/since DBS
dbs_diag = dbs_df.groupby('person_id')[['procedure_date','condition_start_date']].agg('min').reset_index()
dbs_diag['dbs_surgery'] = np.where(
    dbs_diag['procedure_date'].isna(), dbs_diag['condition_start_date'],
    np.where(dbs_diag['condition_start_date'].isna(), dbs_diag['procedure_date'],
             np.minimum(dbs_diag['procedure_date'], dbs_diag['condition_start_date']))
)
dbs_diag.drop(columns=['condition_start_date','procedure_date'], inplace=True)
days_to_dbs = dbs_diag.merge(first_diag, on='person_id', how='inner')
days_to_dbs.rename(columns={'diagnosis_date':'pd_diagnosis'}, inplace=True)
days_to_dbs['days_to_dbs']    = ((days_to_dbs['dbs_surgery'] - days_to_dbs['pd_diagnosis']) / np.timedelta64(1,'D')).astype(int)
days_to_dbs['days_since_dbs'] = ((DATA_CUTOFF - days_to_dbs['dbs_surgery']) / np.timedelta64(1,'D')).astype(int)
days_to_dbs.drop(columns=['dbs_surgery','pd_diagnosis'], inplace=True)
mean_led_per_visit = mean_led_per_visit.merge(days_to_dbs, on='person_id', how='left')
mean_led_per_visit['days_to_dbs'].fillna(days_to_dbs['days_to_dbs'].mean(),       inplace=True)
mean_led_per_visit['days_since_dbs'].fillna(days_to_dbs['days_since_dbs'].mean(), inplace=True)
mean_led_per_visit.drop(columns=['days_to_dbs','days_since_dbs'], inplace=True)

# Diagnosed / POA at K-th visit date
mean_led_per_visit['diagnosed_current_visit'] = mean_led_per_visit.apply(
    lambda row: int(row['person_id'] in
        diagnosis_date[diagnosis_date['diagnosis_date']==row['visit_start_datetime']]['person_id'].values),
    axis=1)
mean_led_per_visit['poa_current_visit'] = mean_led_per_visit.apply(
    lambda row: int(row['person_id'] in
        diagnosis_date[
            (diagnosis_date['condition_poa']==1.0) &
            (diagnosis_date['diagnosis_date']==row['visit_start_datetime'])
        ]['person_id'].values),
    axis=1)

print(f"After all merges: {mean_led_per_visit['person_id'].nunique()} patients, "
      f"{len(mean_led_per_visit):,} rows")
nan_cols = mean_led_per_visit.isnull().sum()
nan_cols = nan_cols[nan_cols>0].sort_values(ascending=False)
if len(nan_cols):
    print(f"NaN cols: {nan_cols.to_dict()}")
else:
    print("No NaN columns")


After all merges: 631 patients, 631 rows
NaN cols: {'mean_led_per_visit': 423, 'race_source_value': 2, 'ethnicity_source_value': 2}


## 10 · Home meds

> Set to `0` — requires `Results_UPDATED.csv` which is not loaded here.

In [10]:
mean_led_per_visit['home_meds'] = 0

## 11 · Drug one-hot encoding (K-th visit only)

In [11]:
# Get only PD drug records from the K-th visit for each patient
kth_visit_keys = mean_led_per_visit[['person_id','visit_start_datetime']].copy()

led_df_kth = (
    led_df[led_df['generic_name'].isin(pd_drug_names)]
          .merge(kth_visit_keys, on=['person_id','visit_start_datetime'], how='inner')
)

aggregations = {'generic_name': lambda x: ', '.join(x)}
result = led_df_kth.groupby('person_id').agg(aggregations).reset_index()
result = result.merge(kth_visit_keys, on='person_id', how='left')

unique_drug_sets = [
    ', '.join(sorted({g for g in row.split(', ')}))
    for row in result['generic_name']
]
patient_drugs = pd.DataFrame({
    'person_id':            result['person_id'].values,
    'visit_start_datetime': result['visit_start_datetime'].values,
    'drugs_per_visit':      unique_drug_sets,
})

drugs_list    = patient_drugs['drugs_per_visit'].str.split(', ')
one_hot_drugs = pd.get_dummies(drugs_list.apply(pd.Series).stack()).groupby(level=0).sum()
one_hot_drugs['person_id']            = patient_drugs['person_id'].values
one_hot_drugs['visit_start_datetime'] = patient_drugs['visit_start_datetime'].values

drug_cols = [c for c in one_hot_drugs.columns if c not in ['person_id','visit_start_datetime']]
mean_led_per_visit = mean_led_per_visit.merge(
    one_hot_drugs, on=['person_id','visit_start_datetime'], how='left'
)
mean_led_per_visit[drug_cols] = mean_led_per_visit[drug_cols].fillna(0)
print(f"Drug one-hot columns added: {len(drug_cols)}")


Drug one-hot columns added: 13


## 12 · Final cleanup

In [12]:
mean_led_per_visit.drop(
    columns=['change_in_ledd','is_first_visit','percent_change','is_change'],
    inplace=True, errors='ignore',
)
mean_led_per_visit.drop(columns=['person_id','visit_start_datetime'], inplace=True)
mean_led_per_visit.dropna(inplace=True)

print(f"Final feature matrix: {mean_led_per_visit.shape}")
print(f"Class balance: {mean_led_per_visit['prediction'].value_counts(normalize=True).to_dict()}")


Final feature matrix: (206, 27)
Class balance: {0: 1.0}


## 13 · Preprocessing → `xgboost_df`

In [13]:
xgboost_df = mean_led_per_visit.copy()

# Extract regression target now — perfectly index-aligned
normalized_pctg_change = xgboost_df.pop('normalized_percent_change')

demographic_vars = ['gender_source_value','race_source_value','ethnicity_source_value']
xgboost_df = pd.get_dummies(xgboost_df,
    columns=[v for v in demographic_vars if v in xgboost_df.columns])

scaler       = MinMaxScaler()
numeric_vars = ['mean_led_per_visit','age','length_of_stay',
                'days_since_last_visit','days_to_diagnosis']
for var in [v for v in numeric_vars if v in xgboost_df.columns]:
    xgboost_df[var] = scaler.fit_transform(xgboost_df[[var]])

prediction_col = xgboost_df.pop('prediction')
xgboost_df['prediction'] = prediction_col

normalized_pctg_change = normalized_pctg_change.reindex(xgboost_df.index)

X = xgboost_df.iloc[:,:-1]
y = xgboost_df.iloc[:,-1]

print(f"xgboost_df : {xgboost_df.shape}  |  features: {X.shape[1]}")
print(f"normalized_pctg_change NaN: {normalized_pctg_change.isna().sum()} (should be 0)")


xgboost_df : (206, 31)  |  features: 30
normalized_pctg_change NaN: 0 (should be 0)


## 14 · Stage 1 — XGBoost Classifier

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
d_train = xgb.DMatrix(X_train, label=y_train)
d_test  = xgb.DMatrix(X_test,  label=y_test)

binary_params = {
    'alpha':0, 'lambda':0.1, 'learning_rate':0.01, 'max_depth':7,
    'eval_metric':'auc', 'objective':'binary:logistic',
    'sampling_method':'gradient_based', 'tree_method':'hist', 'device':'cuda',
}

model_s1 = xgb.train(
    binary_params, d_train,
    num_boost_round=800,
    evals=[(d_test,'test')],
    verbose_eval=100,
    early_stopping_rounds=50,
)

y_pred_proba = model_s1.predict(d_test, iteration_range=(0, model_s1.best_iteration+1))
y_pred       = (y_pred_proba > 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)
print(f"\n[Stage 1 · K={K}]  AUC={auc:.4f}  |  ACC={acc:.4f}")
print(classification_report(y_test, y_pred))

# Full-dataset probabilities for Stage 2 filtering
d_all            = xgb.DMatrix(X)
y_pred_proba_all = model_s1.predict(d_all, iteration_range=(0, model_s1.best_iteration+1))

# Youden index threshold from this model's ROC
fpr_s1, tpr_s1, thresholds_s1 = roc_curve(y_test, y_pred_proba)
STAGE2_CUTOFF = float(thresholds_s1[np.argmax(tpr_s1 - fpr_s1)])
print(f"\nYouden threshold: {STAGE2_CUTOFF:.4f}")
print(f"Stage 2 pool at threshold: {(y_pred_proba_all > STAGE2_CUTOFF).sum():,} "
      f"({(y_pred_proba_all > STAGE2_CUTOFF).mean():.1%})")


[0]	test-auc:nan
[50]	test-auc:nan

[Stage 1 · K=1]  AUC=nan  |  ACC=1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        42

    accuracy                           1.00        42
   macro avg       1.00      1.00      1.00        42
weighted avg       1.00      1.00      1.00        42


Youden threshold: inf
Stage 2 pool at threshold: 0 (0.0%)


## 15 · Stage 2 — XGBoost Regressor

In [15]:
xgboost_df['prediction'] = normalized_pctg_change.values
prob_series = pd.Series(y_pred_proba_all, index=xgboost_df.index)
filtered_df = xgboost_df[prob_series > STAGE2_CUTOFF]

print(f"Stage 2 pool: {len(filtered_df):,} rows  ({len(filtered_df)/len(xgboost_df):.1%})")

X2 = filtered_df.iloc[:,:-1]
y2 = filtered_df.iloc[:,-1]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=RANDOM_STATE
)
d2_train = xgb.DMatrix(X2_train, label=y2_train)
d2_test  = xgb.DMatrix(X2_test,  label=y2_test)

regression_params = {
    'alpha':0.1, 'lambda':1, 'learning_rate':0.1, 'max_depth':7,
    'eval_metric':'rmse', 'objective':'reg:squarederror',
    'sampling_method':'gradient_based', 'tree_method':'hist', 'device':'cuda',
}

model_s2 = xgb.train(
    regression_params, d2_train,
    num_boost_round=700,
    evals=[(d2_train,'train'),(d2_test,'test')],
    verbose_eval=100,
    early_stopping_rounds=10,
)

y2_pred = model_s2.predict(d2_test, iteration_range=(0, model_s2.best_iteration+1))
rmse = np.sqrt(mean_squared_error(y2_test, y2_pred))
mae  = mean_absolute_error(y2_test, y2_pred)
r2   = r2_score(y2_test, y2_pred)
print(f"\n[Stage 2 · K={K}]  RMSE={rmse:.4f}  |  MAE={mae:.4f}  |  R²={r2:.4f}")


Stage 2 pool: 0 rows  (0.0%)


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 16 · Summary

In [ ]:
print("=" * 50)
print(f"  K                : {K}")
print(f"  Patients in model: {len(xgboost_df):,}")
print(f"  Stage 1  AUC     : {auc:.4f}")
print(f"  Stage 1  ACC     : {acc:.4f}")
print(f"  Stage 2  RMSE    : {rmse:.4f}")
print(f"  Stage 2  MAE     : {mae:.4f}")
print(f"  Stage 2  R²      : {r2:.4f}")
print("=" * 50)
